# 02 Q1: p53 ODE Model and TCGA-BRCA Survival

This notebook answers Q1:

**Can the p53 ODE model predict patient survival in TCGA-BRCA?**

TCGA-BRCA provides baseline tumour expression and survival follow-up. It does not directly measure doxorubicin response. Therefore this analysis is prognostic: it tests whether simulated p53 DNA-damage response features are associated with overall survival.

## Modelling Approach

The course template uses a DNA-damage response input called `DDR`. For each sample, selected expression values personalise the p53 DNA-damage response model. The model is then simulated across DDR doses from 0.01 to 1.00.

The template outputs simulated p53 S15 and p53 S46 response scores. Here, the main pre-specified Q1 feature is:

`p53_S46_AUC_across_DDR`

S46 is used as the main feature because p53 S46 is commonly interpreted as a stronger stress/apoptosis-associated p53 response than S15 alone. The AUC summarises response across the full DDR dose range rather than picking one dose after seeing the survival results.

This implementation ports the core template ODE equations and fitted parameter set into the notebook. The original template input table used neuroblastoma examples, so TCGA-BRCA expression is converted into gene-wise ratios relative to the TCGA median before being used as multiplicative model inputs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from scipy.integrate import odeint

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

tcga_path = project_root / "data/processed/tcga_brca_survival_expression_table.csv"
scores_path = project_root / "data/processed/tcga_brca_p53_ode_scores.csv"
results_path = project_root / "tables/q1_p53_ode_tcga_survival_results.csv"
summary_path = project_root / "tables/q1_p53_ode_tcga_feature_summary.csv"
km_figure_path = project_root / "figures/q1_p53_ode_tcga_km.png"
distribution_figure_path = project_root / "figures/q1_p53_ode_tcga_feature_distribution.png"

for folder in [scores_path.parent, results_path.parent, summary_path.parent, km_figure_path.parent, distribution_figure_path.parent]:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:

# Minimal adapted p53 DDR ODE scorer from the course template.
# The input expression values are converted to gene-wise ratios relative to the dataset median.
# These ratios personalise the template's initial conditions or synthesis rates.

p53_model_genes = ["ATM", "CHEK2", "HIPK2", "MDM2", "PPM1D", "SIAH1", "TP53", "WSB1"]
ddr_values = np.linspace(0.01, 1.00, 10)
time_grid = np.linspace(0, 50, 10)

# Selected fitted parameter row ported from the course template parameter table.
# These names match the ODE equations below.
p53_params = {
    "kpatm_dox": 0.7141284576673614,
    "jpatm_dox": 0.01001501738522647,
    "kdpatm_dox": 4.287004777747307,
    "jdpatm_dox": 0.022712818746653787,
    "kpchk2": 39.99520814970714,
    "jpchk2": 0.10021818242876597,
    "kdpchk2": 15.246574360684299,
    "jdpchk2": 0.9995440018783482,
    "kpp53_ATM": 5.63137993400672,
    "kpp53_CHK2": 2.6323732318132933,
    "jpp53_ATM": 0.5414577662095134,
    "jpp53_CHK2": 0.583144727403126,
    "kdpp53a": 0.9757782419741889,
    "jdpp53a": 0.01044114413709567,
    "kpp53a": 35.63339377162273,
    "jpp53a": 0.9420163603515592,
    "kdpp53k": 0.7062534360373444,
    "jdpp53k": 0.19506211836488466,
    "kpsiah1": 0.30036132299082025,
    "jpsiah1": 0.01005366224642904,
    "kdpsiah1": 0.3001326978126625,
    "jdpsiah1": 0.8700047866744796,
    "ksmdm21": 0.6229893448940808,
    "jsmdm21": 0.10106525529238898,
    "kswip11": 0.6810544057024757,
    "jswip11": 0.20393136945804016,
    "kdp53": 2.000114586680669,
    "kdp53a": 0.10000649341988067,
    "jup53": 0.09933440059388587,
    "jup53s": 9.936632242874452,
    "juhipk2": 0.04007319402881099,
}

# Fixed constants and baseline initial conditions from the template.
hcmdm21 = 2
hcwip11 = 2
kdhipk2 = 1.5
kdp53m = 0.077
kdwip1m = 0.66
kdwip1 = 2
kdmdm2m = 0.7
kdmdm2 = 2
kuhipk2 = 0.17
hcdeg = 2
kalg = 0.1

p_p53i_0 = 0.1
p_MDM2m_0 = 0.103
p_p53m_0 = 0.1
WIP1m_0 = 0.0594
WIP1_0 = 0.0277
MDM2_0 = 1
p_ATM_0 = 1
p_SIAH_0 = 1
p_HIPK2_0 = 0.018
p_CHK2_0 = 1
ic_zero = 0

kpsmdm2_base = MDM2_0 * kdmdm2 / p_MDM2m_0
h_MDM2_0 = (MDM2_0 ** hcdeg) / (MDM2_0 ** hcdeg + p53_params["jup53"] ** hcdeg)
kpsp53_base = p53_params["kdp53"] * p_p53i_0 / p_p53m_0 * h_MDM2_0
kpswip1_base = kdwip1 * WIP1_0 / WIP1m_0
p_kshipk2_base = kdhipk2 * p_HIPK2_0 + kuhipk2 * p_SIAH_0 * p_HIPK2_0 / (p53_params["juhipk2"] + p_HIPK2_0)


def make_expression_ratios(data, genes):
    ratios = data[genes].astype(float).copy()
    for gene in genes:
        median_value = ratios[gene].median()
        ratios[gene] = ratios[gene] / median_value
    return ratios.clip(lower=0.05, upper=20)


def sample_initial_state_and_rates(row):
    ksp530 = row["TP53"] * p_p53m_0 * kdp53m

    MDM2m = row["MDM2"] * p_MDM2m_0
    MDM2 = MDM2m * kpsmdm2_base / kdmdm2
    h_MDM2 = (MDM2 ** hcdeg) / (MDM2 ** hcdeg + p53_params["jup53"] ** hcdeg)
    p53i = row["TP53"] * p_p53m_0 * kpsp53_base / p53_params["kdp53"] / h_MDM2
    h_p53i = ((kalg * p53i) ** hcmdm21) / (p53_params["jsmdm21"] ** hcmdm21 + (kalg * p53i) ** hcmdm21)
    ksmdm20 = MDM2m * kdmdm2m - p53_params["ksmdm21"] * h_p53i
    if ksmdm20 < 0:
        ksmdm20 = 1e-9

    WIP1m = row["PPM1D"] * WIP1m_0
    kswip10 = WIP1m * kdwip1m

    SIAH_0 = 0.5 * (row["SIAH1"] + row["WSB1"]) * p_SIAH_0
    kshipk2 = row["HIPK2"] * p_kshipk2_base
    hipk2_term = (
        SIAH_0 ** 2 * kuhipk2 ** 2
        + 2 * SIAH_0 * p53_params["juhipk2"] * kdhipk2 * kuhipk2
        - 2 * SIAH_0 * kshipk2 * kuhipk2
        + p53_params["juhipk2"] ** 2 * kdhipk2 ** 2
        + 2 * p53_params["juhipk2"] * kdhipk2 * kshipk2
        + kshipk2 ** 2
    )
    HIPK2 = (kshipk2 - SIAH_0 * kuhipk2 - p53_params["juhipk2"] * kdhipk2 + np.sqrt(max(hipk2_term, 0))) / (2 * kdhipk2)
    if HIPK2 < 0:
        HIPK2 = 1e-9

    x0 = [
        row["ATM"] * p_ATM_0,
        ic_zero,
        row["TP53"] * p_p53m_0,
        p53i,
        ic_zero,
        ic_zero,
        SIAH_0,
        ic_zero,
        HIPK2,
        WIP1m,
        WIP1m * kpswip1_base / kdwip1,
        row["CHEK2"] * p_CHK2_0,
        ic_zero,
        MDM2m,
        MDM2,
    ]
    rates = {
        "ksp530": ksp530,
        "kpsmdm2": kpsmdm2_base,
        "kpsp53": kpsp53_base,
        "kpswip1": kpswip1_base,
        "kshipk2": kshipk2,
        "kswip10": kswip10,
        "ksmdm20": ksmdm20,
    }
    return x0, rates


def p53_ode(x, t, ddr, rates):
    WIP1a = 1

    ATM, ATMp, p53m, p53i, p53s15, p53s46, SIAH1, SIAH1p, HIPK2, WIP1m, WIP1, CHK2, CHK2p, MDM2m, MDM2 = x
    ku = np.heaviside(t, 1)

    v1 = p53_params["kpatm_dox"] * ATM * ddr * ku / (p53_params["jpatm_dox"] + ATM)
    v2 = p53_params["kdpatm_dox"] * WIP1a * ATMp / (p53_params["jdpatm_dox"] + ATMp)
    v3 = p53_params["kpchk2"] * ATMp * CHK2 / (p53_params["jpchk2"] + CHK2)
    v4 = p53_params["kdpchk2"] * WIP1 * CHK2p / (p53_params["jdpchk2"] + CHK2p)
    v5 = p53_params["kpp53_CHK2"] * CHK2p * p53i / (p53_params["jpp53_CHK2"] + p53i)
    v6 = p53_params["kpp53_ATM"] * ATMp * p53i / (p53_params["jpp53_ATM"] + p53i)
    v7 = p53_params["kdpp53a"] * WIP1 * p53s15 / (p53_params["jdpp53a"] + p53s15)
    v8 = p53_params["kpp53a"] * HIPK2 * p53s15 / (p53_params["jpp53a"] + p53s15)
    v9 = p53_params["kdpp53k"] * p53s46 / (p53_params["jdpp53k"] + p53s46)
    v10 = p53_params["kpsiah1"] * ATMp * SIAH1 / (p53_params["jpsiah1"] + SIAH1)
    v11 = p53_params["kdpsiah1"] * SIAH1p / (p53_params["jdpsiah1"] + SIAH1p)

    p53alg = kalg * p53i + p53s15 + p53s46
    p53s15tot = p53s15 + p53s46
    v12 = rates["ksp530"]
    v13 = rates["ksmdm20"]
    v14 = p53_params["ksmdm21"] * (p53alg ** hcmdm21) / (p53_params["jsmdm21"] ** hcmdm21 + p53alg ** hcmdm21)
    v15 = rates["kswip10"]
    v16 = p53_params["kswip11"] * (p53s15tot ** hcwip11) / (p53_params["jswip11"] ** hcwip11 + p53s15tot ** hcwip11)

    v21 = rates["kpsp53"] * p53m
    v22 = rates["kshipk2"]
    v23 = rates["kpswip1"] * WIP1m
    v24 = rates["kpsmdm2"] * MDM2m

    v25 = p53_params["kdp53"] * p53i * (MDM2 ** hcdeg) / (MDM2 ** hcdeg + p53_params["jup53"] ** hcdeg)
    v26 = p53_params["kdp53a"] * p53s15 * (MDM2 ** hcdeg) / (MDM2 ** hcdeg + p53_params["jup53s"] ** hcdeg)
    v27 = p53_params["kdp53a"] * p53s46 * (MDM2 ** hcdeg) / (MDM2 ** hcdeg + p53_params["jup53s"] ** hcdeg)
    v28 = kdhipk2 * HIPK2
    v29 = kdp53m * p53m
    v30 = kdwip1m * WIP1m
    v31 = kdwip1 * WIP1
    v32 = kdmdm2m * MDM2m
    v33 = kdmdm2 * MDM2
    v36 = kuhipk2 * SIAH1 * HIPK2 / (p53_params["juhipk2"] + HIPK2)

    return [
        -v1 + v2,
        v1 - v2,
        v12 - v29,
        -v5 - v6 + v7 + v21 - v25,
        v5 + v6 - v7 - v8 + v9 - v26,
        v8 - v9 - v27,
        -v10 + v11,
        v10 - v11,
        v22 - v28 - v36,
        v15 + v16 - v30,
        v23 - v31,
        -v3 + v4,
        v3 - v4,
        v13 + v14 - v32,
        v24 - v33,
    ]


def simulate_p53_scores(data, id_columns):
    ratios = make_expression_ratios(data, p53_model_genes)
    records = []

    for index, row in ratios.iterrows():
        x0, rates = sample_initial_state_and_rates(row)
        s15_values = []
        s46_values = []

        for ddr in ddr_values:
            solution = odeint(p53_ode, x0, time_grid, args=(ddr, rates))
            s15_values.append(solution[-1, 4])
            s46_values.append(solution[-1, 5])

        record = {col: data.loc[index, col] for col in id_columns if col in data.columns}
        record["p53_S15_low_DDR"] = s15_values[0]
        record["p53_S15_high_DDR"] = s15_values[-1]
        record["p53_S15_AUC_across_DDR"] = np.trapezoid(s15_values, ddr_values)
        record["p53_S46_low_DDR"] = s46_values[0]
        record["p53_S46_high_DDR"] = s46_values[-1]
        record["p53_S46_AUC_across_DDR"] = np.trapezoid(s46_values, ddr_values)
        record["p53_S46_to_S15_ratio"] = record["p53_S46_AUC_across_DDR"] / (record["p53_S15_AUC_across_DDR"] + 1e-9)
        records.append(record)

    return pd.DataFrame(records)


## Load TCGA-BRCA Expression and Survival

The p53 model uses the eight core input genes from the course template:

`ATM`, `CHEK2`, `HIPK2`, `MDM2`, `PPM1D`, `SIAH1`, `TP53`, `WSB1`.

Age is used later as a simple adjusted covariate. Stage is left for later sensitivity analysis because the labels need extra cleaning before they can be modelled cleanly.

In [ ]:
tcga = pd.read_csv(tcga_path)
required_columns = ["patient_id", "overall_survival_time", "overall_survival_event"] + p53_model_genes
missing_columns = [col for col in required_columns if col not in tcga.columns]

print("TCGA-BRCA table shape:", tcga.shape)
print("Missing required columns:", missing_columns)
print("Age available:", "age" in tcga.columns)
print("Stage available:", "stage" in tcga.columns)

if missing_columns:
    raise ValueError("Missing required TCGA columns for Q1 p53 ODE analysis.")

tcga_q1 = tcga.dropna(subset=required_columns).copy()
tcga_q1["overall_survival_time"] = pd.to_numeric(tcga_q1["overall_survival_time"], errors="coerce")
tcga_q1["overall_survival_event"] = pd.to_numeric(tcga_q1["overall_survival_event"], errors="coerce")
tcga_q1 = tcga_q1[tcga_q1["overall_survival_time"] > 0].copy()
tcga_q1["overall_survival_event"] = tcga_q1["overall_survival_event"].astype(int)

print("Patients available for Q1:", len(tcga_q1))
print("Deaths/events:", int(tcga_q1["overall_survival_event"].sum()))
print("Censored:", int((tcga_q1["overall_survival_event"] == 0).sum()))
display(tcga_q1[["patient_id", "overall_survival_time", "overall_survival_event", "age"] + p53_model_genes[:4]].head())

## Generate TCGA p53 ODE Scores

Each patient is scored by simulating the p53 DDR model across the same DDR doses used in the course template. The output table contains low-dose, high-dose and AUC summaries for S15 and S46.

In [ ]:
tcga_scores = simulate_p53_scores(tcga_q1, id_columns=["patient_id", "sample_id"])
tcga_scores.to_csv(scores_path, index=False)

print("Saved TCGA p53 ODE scores to:", scores_path.relative_to(project_root))
print("Score table shape:", tcga_scores.shape)
display(tcga_scores.head())

In [ ]:
p53_feature_cols = [
    "p53_S15_low_DDR",
    "p53_S15_high_DDR",
    "p53_S15_AUC_across_DDR",
    "p53_S46_low_DDR",
    "p53_S46_high_DDR",
    "p53_S46_AUC_across_DDR",
    "p53_S46_to_S15_ratio",
]
main_p53_feature = "p53_S46_AUC_across_DDR"

feature_summary = tcga_scores[p53_feature_cols].describe().T.reset_index().rename(columns={"index": "feature"})
feature_summary.to_csv(summary_path, index=False)
print("Saved Q1 feature summary to:", summary_path.relative_to(project_root))
display(feature_summary)

## Q1 Survival Tests

The survival analysis is kept simple:

1. univariate Cox model for each p53 ODE feature;
2. age-adjusted Cox model for the pre-specified main feature if age is available;
3. median high/low Kaplan-Meier split for the main feature.

The main feature is pre-specified as `p53_S46_AUC_across_DDR`.

For Cox models, p53 ODE features are standardised first, so hazard ratios are reported per one standard deviation increase in the feature. The Kaplan-Meier median split uses the raw score.

In [ ]:
survival_data = tcga_q1[["patient_id", "overall_survival_time", "overall_survival_event", "age"]].merge(
    tcga_scores[["patient_id"] + p53_feature_cols],
    on="patient_id",
    how="inner",
)

q1_rows = []

for feature in p53_feature_cols:
    z_feature = feature + "_z"
    model_data = survival_data[["overall_survival_time", "overall_survival_event", feature]].dropna().copy()
    model_data[z_feature] = (model_data[feature] - model_data[feature].mean()) / model_data[feature].std()

    cox = CoxPHFitter()
    cox.fit(model_data[["overall_survival_time", "overall_survival_event", z_feature]], duration_col="overall_survival_time", event_col="overall_survival_event")
    row = cox.summary.loc[z_feature]
    q1_rows.append({
        "analysis": "univariate_cox_per_1_sd",
        "feature": feature,
        "coef": row["coef"],
        "hazard_ratio": row["exp(coef)"],
        "ci_lower": row["exp(coef) lower 95%"],
        "ci_upper": row["exp(coef) upper 95%"],
        "p_value": row["p"],
        "n_patients": len(model_data),
        "n_events": int(model_data["overall_survival_event"].sum()),
    })

if "age" in survival_data.columns and survival_data["age"].notna().sum() > 0:
    age_data = survival_data[["overall_survival_time", "overall_survival_event", main_p53_feature, "age"]].dropna().copy()
    main_z_feature = main_p53_feature + "_z"
    age_data[main_z_feature] = (age_data[main_p53_feature] - age_data[main_p53_feature].mean()) / age_data[main_p53_feature].std()

    age_cox = CoxPHFitter()
    age_cox.fit(age_data[["overall_survival_time", "overall_survival_event", main_z_feature, "age"]], duration_col="overall_survival_time", event_col="overall_survival_event")
    for term, row in age_cox.summary.iterrows():
        reported_feature = main_p53_feature if term == main_z_feature else term
        q1_rows.append({
            "analysis": "age_adjusted_cox_main_feature_per_1_sd",
            "feature": reported_feature,
            "coef": row["coef"],
            "hazard_ratio": row["exp(coef)"],
            "ci_lower": row["exp(coef) lower 95%"],
            "ci_upper": row["exp(coef) upper 95%"],
            "p_value": row["p"],
            "n_patients": len(age_data),
            "n_events": int(age_data["overall_survival_event"].sum()),
        })

median_score = survival_data[main_p53_feature].median()
survival_data["p53_main_feature_group"] = np.where(
    survival_data[main_p53_feature] >= median_score,
    "High p53 S46 AUC",
    "Low p53 S46 AUC",
)
high_group = survival_data[survival_data["p53_main_feature_group"] == "High p53 S46 AUC"]
low_group = survival_data[survival_data["p53_main_feature_group"] == "Low p53 S46 AUC"]
logrank = logrank_test(
    high_group["overall_survival_time"],
    low_group["overall_survival_time"],
    event_observed_A=high_group["overall_survival_event"],
    event_observed_B=low_group["overall_survival_event"],
)
q1_rows.append({
    "analysis": "median_high_vs_low_logrank",
    "feature": main_p53_feature,
    "coef": np.nan,
    "hazard_ratio": np.nan,
    "ci_lower": np.nan,
    "ci_upper": np.nan,
    "p_value": logrank.p_value,
    "n_patients": len(survival_data),
    "n_events": int(survival_data["overall_survival_event"].sum()),
})

q1_results = pd.DataFrame(q1_rows)
q1_results["cox_feature_scale"] = np.where(q1_results["analysis"].str.contains("cox"), "per 1 standard deviation for p53 features", "not applicable")
q1_results.to_csv(results_path, index=False)
print("Saved Q1 survival results to:", results_path.relative_to(project_root))
display(q1_results)

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.hist(survival_data[main_p53_feature].dropna(), bins=30, edgecolor="black", color="#4c78a8")
plt.axvline(median_score, color="black", linestyle="--", linewidth=1, label="Median")
plt.xlabel(main_p53_feature)
plt.ylabel("Number of TCGA-BRCA patients")
plt.title("Distribution of TCGA p53 S46 DDR response score")
plt.legend()
plt.tight_layout()
plt.savefig(distribution_figure_path, dpi=200)
plt.show()
print("Saved feature distribution figure to:", distribution_figure_path.relative_to(project_root))

In [ ]:
kmf = KaplanMeierFitter()
plt.figure(figsize=(7, 5))
for group_name, group_data in survival_data.groupby("p53_main_feature_group"):
    kmf.fit(
        group_data["overall_survival_time"],
        event_observed=group_data["overall_survival_event"],
        label=f"{group_name} (n={len(group_data)})",
    )
    kmf.plot_survival_function(ci_show=False)

plt.xlabel("Overall survival time (days)")
plt.ylabel("Survival probability")
plt.title("TCGA-BRCA survival by p53 S46 DDR response score")
plt.text(
    0.03,
    0.08,
    f"Median split\nLog-rank p = {logrank.p_value:.3g}",
    transform=plt.gca().transAxes,
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "lightgray"},
)
plt.tight_layout()
plt.savefig(km_figure_path, dpi=200)
plt.show()
print("Saved Q1 KM figure to:", km_figure_path.relative_to(project_root))

## Q1 Interpretation

Use the saved Cox and Kaplan-Meier outputs to report whether the pre-specified p53 S46 AUC feature is associated with TCGA-BRCA overall survival.

This is a prognostic analysis. TCGA-BRCA does not directly measure doxorubicin response, so the result should not be described as evidence of chemotherapy sensitivity. The ODE model is adapted from the course template and depends on that model's biological assumptions.